In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from tqdm.auto import tqdm
from torchvision import transforms
from torchvision.datasets import MNIST # Training dataset
from torchvision.utils import make_grid
from torch.utils.data import DataLoader

# Dataset


- Segmentar os pulmões nas imagens
- Utilizar como entrada da rede gerativa

# Segmentação

In [ ]:
print('hello world')

hello world


# Modelo

In [ ]:
def encoder_block(in_dim,out_dim):
  return nn.Sequential(nn.Conv2d(in_channels=in_dim, out_channels=out_dim, kernel_size=4, stride=2, padding=1, dilation=1, bias=True),
                        nn.BatchNorm2d(in_channels=out_dim),
                        nn.LeakyReLU(alpha=0.2,inplace=True))

In [ ]:
def decoder_block(in_dim,out_dim):
  return nn.Sequential(nn.ConvTranspose2d(in_channels=in_dim, out_channels=out_dim, kernel_size=4, stride=2, padding=1, output_padding=0, groups=1, bias=True, dilation=1),
                       nn.BatchNorm2d(in_channels=out_dim),
                       nn.Dropout(p=0.5),
                       nn.ReLU(inplace=True))

In [ ]:
class Generador(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Sequential(nn.Conv2d(in_channels=1, out_channels=64, kernel_size=4, stride=2, padding=1, dilation=1, bias=True),
                               nn.LeakyReLU(alpha=0.2,inplace=True))
    self.conv2 = encoder_block(in_dim=64,out_dim=128)
    self.conv3 = encoder_block(in_dim=128,out_dim=256)
    self.conv4 = encoder_block(in_dim=256,out_dim=512)
    self.conv5 = encoder_block(in_dim=512,out_dim=512)
    self.conv6 = encoder_block(in_dim=512,out_dim=512)
    self.conv7 = encoder_block(in_dim=512,out_dim=512)
    self.conv8 = encoder_block(in_dim=512,out_dim=512)
    self.deconv8 = decoder_block(in_dim=512,out_dim=512)
    self.deconv7 = decoder_block(in_dim=512,out_dim=512)
    self.deconv6 = decoder_block(in_dim=512,out_dim=512)
    self.deconv5 = decoder_block(in_dim=512,out_dim=512)
    self.deconv4 = decoder_block(in_dim=512,out_dim=256)
    self.deconv3 = decoder_block(in_dim=256,out_dim=128)
    self.deconv2 = decoder_block(in_dim=128,out_dim=64)
    self.deconv1 = nn.Sequential(nn.ConvTranspose2d(in_channels=64, out_channels=1, kernel_size=4, stride=2, padding=1, output_padding=0, groups=1, bias=True, dilation=1),
                       nn.Tanh())

  def forward(self,x):
    enc1 = self.conv1(x) #64
    enc2 = self.conv2(enc1) #128
    enc3 = self.conv3(enc2) #256
    enc4 = self.conv4(enc3) #512
    enc5 = self.conv5(enc4) #512
    enc6 = self.conv6(enc5) #512
    enc7 = self.conv7(enc6) #512
    enc8 = self.conv8(enc7) #512
    dec8 = self.deconv8(enc8)+enc7
    dec7 = self.deconv7(dec8)+enc6
    dec6 = self.deconv6(dec7)+enc5
    dec5 = self.deconv5(dec6)+enc4
    dec4 = self.deconv4(dec5)+enc3
    dec3 = self.deconv3(dec4)+enc2
    dec2 = self.deconv2(dec3)+enc1
    dec1 = self.deconv1(dec2)
    return dec1

  def get_gen(self):
      return self.gen

In [ ]:
def block_discriminator(in_dim,out_dim):
  return nn.Sequential(nn.Conv2d(in_channels=in_dim, out_channels=out_dim, kernel_size=4, stride=2, padding=1, dilation=1, bias=True),
                       nn.LeakyReLU(alpha=0.2,inplace=True))

In [ ]:
class Discriminador(nn.Module):
  def __init__(self):
      super().__init__()
      self.conv1 = block_discriminator(in_dim=1,out_dim=64)
      self.conv2 = block_discriminator(in_dim=64,out_dim=128)
      self.conv3 = block_discriminator(in_dim=128,out_dim=256)
      self.conv4 = nn.Sequential(nn.Conv2d(in_channels=256, out_channels=512, kernel_size=4, stride=1, padding=1, dilation=1, bias=True),
                       nn.LeakyReLU(alpha=0.2,inplace=True))
      self.conv5 = nn.Sequential(nn.Conv2d(in_channels=512, out_channels=1, kernel_size=4, stride=1, padding=1, dilation=1, bias=True),
                       nn.Sigmoid())

  def forward(self,x):
      x = self.conv1(x)
      x = self.conv2(x)
      x = self.conv3(x)
      x = self.conv4(x)
      out = self.conv5(x)
      return out


  def get_disc(self):
        return self.disc

# Loss e Métricas

In [ ]:
def disc_loss(gen,disc,criterion,input_mask,input_img,device):
    gen_img = gen(input_mask).detach()
    ans_gen = disc(gen_img)
    gt_gen = torch.zeros_like(ans_gen)
    ans_real = disc(input_img)
    gt_real = torch.ones_like(ans_real)
    x = torch.cat((ans_real.reshape(-1),ans_gen.reshape(-1)))
    y = torch.cat((gt_real.reshape(-1),gt_gen.reshape(-1)))
    loss = criterion(x,y)
    #the regularization (l1 norm) is not important here: is independent of D
    return loss.mean()

def gen_loss(gen,disc,criterion,input_mask,input_img,regularization,device):
    gen_img = gen(input_mask).detach()
    ans_gen = disc(gen_img)
    #we want ans_gen close to 1: to trick the disc
    gt_gen = torch.ones_like(ans_gen)
    loss_pt1 = criterion(ans_gen,gt_gen).mean()
    loss_pt2 = torch.sum(torch.abs(input_img-gen_img),dim=(1,2)).mean()
    return loss_pt1+regularization*loss_pt2

# Treinamento

In [ ]:
# Set your parameters
criterion = nn.BCELoss()
n_epochs = 1
z_dim = 64
display_step = 500
batch_size = 128
lr = 0.00001

# Load MNIST dataset as tensors
dataloader = DataLoader(
    MNIST('.', download=True, transform=transforms.ToTensor()),
    batch_size=batch_size,
    shuffle=True)

### DO NOT EDIT ###
device = 'cuda'

Failed to download (trying next):
HTTP Error 403: Forbidden



100.0%


Extracting ./MNIST/raw/train-images-idx3-ubyte.gz to ./MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100.0%


Extracting ./MNIST/raw/train-labels-idx1-ubyte.gz to ./MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100.0%


Extracting ./MNIST/raw/t10k-images-idx3-ubyte.gz to ./MNIST/raw

Failed to download (trying next):
HTTP Error 403: Forbidden



100.0%

Extracting ./MNIST/raw/t10k-labels-idx1-ubyte.gz to ./MNIST/raw



In [ ]:
gen = Generator().to(device)
gen_opt = torch.optim.Adam(gen.parameters(), lr=lr)
disc = Discriminator().to(device)
disc_opt = torch.optim.Adam(disc.parameters(), lr=lr)